# 06 Grad-CAM 与错误案例分析

**目标：** 利用已训练模型与真实逐图预测，分析域内/真实场景的正确与错误案例，观察模型关注区域。

**前置条件：** 已生成所选 run 的 `best.pt` 与 `predictions.csv`；Grad-CAM 需要 `requirements-demo.txt` 中的可选依赖。

**输入：** 明确选定的实验目录、预测记录及对应原图。

**输出：** `outputs/figures/<run_id>/gradcam/` 中的热力图和本 Notebook 中的案例记录。

默认关闭分析。Grad-CAM 只表示对类别得分的敏感区域；不能据此证明模型已经理解病斑机理。

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

## 1. 选择真实实验与案例来源

先填写 `ANALYSIS_RUN`。`ANALYSIS_DATASET` 使用 Notebook 05 保存的名称，通常为 `plantvillage_test` 或 `plantdoc_external_test`。初次可查看一张正确和一张错误案例，正式报告应补充具有代表性的类别、场景与错误类型，并说明选择依据。

In [ ]:
RUN_ANALYSIS = False
ANALYSIS_RUN = ""  # 填写 outputs/runs/ 下已训练的实验目录名
ANALYSIS_DATASET = "plantdoc_external_test"
MAX_PER_GROUP = 2

analysis_run_dir = PROJECT_ROOT / "outputs" / "runs" / ANALYSIS_RUN if ANALYSIS_RUN else None
analysis_ready = bool(analysis_run_dir and (analysis_run_dir / "best.pt").is_file()
                      and (analysis_run_dir / "predictions.csv").is_file())
selected_cases = []
if not ANALYSIS_RUN:
    print("请先填写 ANALYSIS_RUN，选择一个真实训练并评价过的实验。")
elif not analysis_ready:
    print("所选实验缺少 best.pt 或 predictions.csv；请完成训练和评价。")
elif not RUN_ANALYSIS:
    print("实验已选择。确认后将 RUN_ANALYSIS 设为 True，再读取与分析预测。")
else:
    prediction_rows = [r for r in read_csv(analysis_run_dir / "predictions.csv")
                       if r.get("dataset") == ANALYSIS_DATASET]
    for correct in (True, False):
        matches = [r for r in prediction_rows
                   if str(r.get("correct", "")).lower() == str(correct).lower()]
        selected_cases.extend(matches[:MAX_PER_GROUP])
        print("正确案例" if correct else "错误案例", "总数：", len(matches))
    show_table(selected_cases)
    if not selected_cases:
        print("未找到所选数据集的预测记录，请核对 ANALYSIS_DATASET 与 Notebook 05 的评价结果。")

## 2. 生成真实 Grad-CAM

使用模型预测类别作为解释目标，并以保存的确定性预处理处理原图。热力图只在本单元实际运行后生成；不会预置或模拟分析图片。

In [ ]:
if RUN_ANALYSIS and analysis_ready and selected_cases:
    from PIL import Image
    from IPython.display import display
    from src.data import build_transforms
    from src.evaluate import load_trained_model
    from src.explain import gradcam_image
    from src.utils import get_device

    device = get_device(BASE.get("device", "auto"))
    model, checkpoint = load_trained_model(analysis_run_dir, device=device)
    data_config = checkpoint["config"]["data"]
    transform = build_transforms("none", **{k: data_config[k]
                                 for k in ("image_size", "mean", "std") if k in data_config})
    figure_dir = PROJECT_ROOT / "outputs" / "figures" / ANALYSIS_RUN / "gradcam"
    figure_dir.mkdir(parents=True, exist_ok=True)
    for case_index, case in enumerate(selected_cases, start=1):
        image_path = PROJECT_ROOT / case["image_path"]
        with Image.open(image_path) as source:
            image = source.convert("RGB")
        overlay = gradcam_image(model, checkpoint["model_name"], image, transform,
                                target_class=int(case["predicted_index"]), device=device)
        output_path = figure_dir / f"{ANALYSIS_DATASET}_{case_index:02d}_{image_path.stem}.png"
        overlay.save(output_path)
        print("原图：", case["image_path"])
        print("真实：", case["true_label"], "预测：", case["predicted_label"],
              "置信度：", round(float(case["confidence"]), 4))
        display(image.resize((224, 224)), overlay)
        print("热力图已保存：", output_path)
else:
    print("尚未生成热力图：先选择实验、数据集，并开启 RUN_ANALYSIS。")

## 结论记录

逐例记录，不用少量示例代替整体评价：

| 案例路径 | 真实类别/预测类别 | 场景与病斑特点 | 注意区域 | 可能错误原因 |
|---|---|---|---|---|
| 待实际分析后填写 | | | | |

- 案例选择依据与覆盖范围：待填写。
- 正确样本是否也关注背景：待填写。
- 错误与模糊、遮挡、光照、相似病斑、标签质量的关系：待填写。
- Grad-CAM 的解释限制：待填写。